 ## GPT -> Generally Pretrained Transformer

In [1]:
with open ('input.txt','r',encoding='utf-8') as f:
    text = f.read()

In [2]:
print("length of dataset in char: " , len(text))

length of dataset in char:  1115394


In [3]:
# first 1k chars
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [4]:
# building vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [5]:
# Vectorization
# mapping characters -> integers ( char level vectorization)

stoi = {ch: i for i,ch in enumerate(chars)} #string to integer
itos = {i: ch for i,ch in enumerate(chars)} # integer to string
encode = lambda s:[stoi[c] for c in s]      # encoder : takes a string and gives output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder : takes a list of integers and output a string

print(encode("hello I am kk"))
print(decode(encode("hello I am kk")))



[46, 43, 50, 50, 53, 1, 21, 1, 39, 51, 1, 49, 49]
hello I am kk


In [6]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000]) # the 1000 characters we looked at earier will to the GPT look like this

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [7]:
# training and testing set
# 90 / 10 
n= int(0.9*len(data))
train_data = data[:n]
val_data = data[n:] 

In [8]:
# cant give all the text at once to the transformer to train on as computationally very expensive and not enough space so will divide into blocks
# making chunks also allow us to process faster as parralelization comes into play -> use of gpu
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [9]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target is: {target}")

when input is tensor([18]) the target is: 47
when input is tensor([18, 47]) the target is: 56
when input is tensor([18, 47, 56]) the target is: 57
when input is tensor([18, 47, 56, 57]) the target is: 58
when input is tensor([18, 47, 56, 57, 58]) the target is: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target is: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target is: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is: 58


In [10]:
# to handle parralelization
torch.manual_seed(1337)
batch_size = 4 # how many seq will the transformer process in parallel
block_size = 8 # what is the max context length for predictions

def get_batch(split):
    # generating small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size , (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x , y
xb , yb = get_batch('train')
print('inputs:' , xb.shape )
print(xb)
print('targets:' , yb.shape )
print(yb)

for b in range(batch_size): #batch dimension
    for t in range(block_size): #time dimension
        context = xb[b,:t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")



inputs: torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets: torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53, 56,

In [11]:
print(xb) # batch of input to be fed to transformer

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


In [ ]:
# simple baseline nn -> bigram language model 
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    '''predicTs the next char based on current char'''
    def __init__(self , vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        # logit is a raw score produced by the model before converting it into probabilities
        self.token_embedding_table = nn.Embedding(vocab_size,vocab_size) #creates a lokup table - we have 65 chars so approx 65*65 params mapped as current char -> probability / logits for next char

    def forward(self , idx , targets=None): #idx contains input characters 
        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) #(B,T,C) Batch,time,channel tensor -> predictions about what come next

        if targets is None:
            loss = None
        else: 
            # pytorch expects b,c,t 
            # b->batch size ; t->time/context dimension ; c-> no of classes / vocab_size
            B,T,C = logits.shape
            logits = logits.view(B*T, C) #preserving channel dimension as second 
            targets = targets.view(B*T) # can also do -1 so pytorch automatically figures out what it should be 
            # loss -> negative log likelihood (cross entropy) 
            loss = F.cross_entropy(logits,targets) # -ve log likelihood
            

        return logits , loss
    
    def generate (self,idx , max_new_tokens):
        '''takes all the preceding vallues and generate -> not good for bigram but is kinda generalised (comments not docs xD)'''
        # idx is (B,T) array of indices in the current context 
        for _ in range(max_new_tokens):
            # get the prediction
            logits , loss = self(idx)
            # focus only on the last time step
            logits = logits[:,-1,:] #becomes (B,C)
            # apply softmax to get probabilities 
            probs = F.softmax(logits , dim = -1) #(B,C)
            # sample from the distribution
            idx_next = torch.multinomial(probs , num_samples=1) #(B,1)
            # append sampled index to the running sequence
            idx = torch.cat((idx,idx_next),dim=1) # (B,T+1)
        return idx  
             



m = BigramLanguageModel(vocab_size)
logits,loss = m(xb,yb)
print(logits.shape)
print(loss)


print(decode(m.generate(idx = torch.zeros((1,1) , dtype = torch.long), max_new_tokens=100)[0].tolist() )) #generating with this totally random model -> so obv the generation wouldn't be any good

torch.Size([32, 65])


tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [13]:
# Creating PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)  

In [ ]:
batch_size = 32
for steps in range(10000):
    # sample a batch of data
    xb , yb = get_batch('train')

    #evaluate the loss
    logits , loss = m(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    print(loss.item()) 

4.692410945892334
4.664144515991211
4.7657151222229
4.70655632019043
4.5956573486328125
4.7101240158081055
4.713661193847656
4.686909198760986
4.700076103210449
4.7182841300964355
4.715603351593018
4.684308052062988
4.745601177215576
4.735717296600342
4.666238784790039
4.58615255355835
4.714625835418701
4.671982765197754
4.715047359466553
4.74489164352417
4.630162715911865
4.707578182220459
4.670665264129639
4.582583427429199
4.739546298980713
4.674807071685791
4.805595874786377
4.749917507171631
4.691989421844482
4.604404926300049
4.721841335296631
4.741591930389404
4.6099629402160645
4.662769794464111
4.730099678039551
4.738433361053467
4.688235282897949
4.639987945556641
4.736632823944092
4.709773540496826
4.736939430236816
4.69184684753418
4.719646453857422
4.752516746520996
4.570086479187012
4.643786907196045
4.699163913726807
4.806960105895996
4.572142601013184
4.717066287994385
4.509502410888672
4.603540897369385
4.6649675369262695
4.712099075317383
4.736576557159424
4.812878131

In [15]:
print(decode(m.generate(idx = torch.zeros((1,1),dtype = torch.long),max_new_tokens=500)[0].tolist()))


lso br. ave aviasurf my, yxMPZI ivee iuedrd whar ksth y h bora s be hese, woweee; the! KI 'de, ulseecherd d o blllando;LUCEO, oraingofof win!
RIfans picspeserer hee tha,
TOFonk? me ain ckntoty ded. bo'llll st ta d:
ELIS me hurf lal y, ma dus pe athouo
BEY:! Indy; by s afreanoo adicererupa anse tecorro llaus a!
OLeneerithesinthengove fal amas trr
TI ar I t, mes, n IUSt my w, fredeeyove
THek' merer, dd
We ntem lud engitheso; cer ize helorowaginte the?
Thak orblyoruldvicee chot, p,
Bealivolde Th li


In [16]:
# tokens are not having any semantic meaning rn